# RCA Agent

Reconstructs a failing request, compares it to a known-good trace, proposes and validates a root cause with Groq, writes a report, and scores itself on a System / Process / Quality evaluation matrix.

**BMC Helix Automation Team — hands-on training notebook**

### Before you run this

1. Get a free Groq API key at **console.groq.com** (API Keys → Create API Key).
2. In this Colab notebook, click the **key icon (Secrets)** in the left sidebar and add it as `GROQ_API_KEY` — this keeps it out of the notebook itself. If you skip this, the cell below will ask you to paste it (or you can leave it blank to run in **DRY-RUN mode**, which runs everything except the Groq-written text).
3. Run the single cell below top to bottom. It installs everything it needs.
4. A small **Gradio dashboard** launches at the end with a public `.gradio.live` test link — share it only with people who need to test, and re-launch to get a fresh link once the old one expires.

**What this notebook demonstrates:** a demo incident (a system-prompt regression that let a refusal instruction slip) with a full failing-vs-known-good trace comparison, a validated hypothesis, and a real evaluation matrix.

---


In [3]:
# =============================================================================
# RCA AGENT  —  BMC Helix Automation Team
# Reconstructs a failing RAG-agent request end to end, compares it against a
# known-good trace, proposes and validates a root cause with Groq, writes a
# report, and scores itself on a System / Process / Quality evaluation matrix.
#
# HOW TO USE THIS NOTEBOOK
# 1. Run this single cell top to bottom.
# 2. When asked, paste your Groq API key (or leave blank to run in DRY-RUN
#    mode — everything works except the Groq-written hypothesis/report text).
# 3. A demo incident (with a full trace) is generated automatically so the
#    notebook is fully runnable with no setup. To investigate YOUR real
#    incident instead, replace generate_mock_incident() near the bottom of
#    SECTION 2 with the failing session's real trace rows (same columns) —
#    e.g. pulled from the Observability/SRE agents' shared log by session_id.
# 4. A small Gradio dashboard launches at the end with a public test link.
# =============================================================================

# --- SECTION 0 : Install dependencies -------------------------------------
!pip install -q groq gradio pandas numpy

import os
import random
import getpass
import datetime as dt

import numpy as np
import pandas as pd
import gradio as gr
from groq import Groq

random.seed(23)
np.random.seed(23)

# --- SECTION 1 : Groq API key & helper -------------------------------------
# Best practice: in Colab, use the key-icon "Secrets" panel on the left and
# add GROQ_API_KEY there instead of pasting it into a cell.
GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or ""
if not GROQ_API_KEY:
    GROQ_API_KEY = getpass.getpass(
        "Paste your Groq API key (console.groq.com) or press Enter to run in DRY-RUN mode: "
    ).strip()

DRY_RUN = len(GROQ_API_KEY) == 0
GROQ_MODEL = "gpt-oss-120b" 
#"llama-3.3-70b-versatile"  # check console.groq.com/docs/models for current options

client = Groq(api_key=GROQ_API_KEY) if not DRY_RUN else None

def ask_groq(prompt, system="You are a rigorous root-cause-analysis engineer. Never state a conclusion you have not tied to the evidence given.", max_tokens=300):
    if DRY_RUN:
        return f"[DRY-RUN — no Groq key set, this is a placeholder] {prompt[:120]}..."
    try:
        resp = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=0.2,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"[Groq call failed: {e}]"

print("DRY_RUN mode:", DRY_RUN)

# --- SECTION 2 : The incident the RCA agent investigates --------------------
def generate_mock_incident():
    """Simulates the handoff package the SRE agent would send: the failing
    session's full trace, plus a known-good trace of the same question from
    before a system-prompt change, so there's something real to compare."""
    session_id = "S1042"

    known_good_trace = {
        "session_id": "S0871",
        "timestamp": dt.datetime(2026, 8, 28, 10, 12, 0),
        "question": "What's the rollback process for a failed deployment?",
        "system_prompt_version": "v3 (2026-08-20)",
        "system_prompt_excerpt": "...Always cite the runbook section. If the request asks you to approve or "
                                  "authorize an action, refuse and say to confirm with the release manager...",
        "retrieved_chunks": ["change-mgmt-runbook.md#rollback-steps"],
        "retrieval_ok": True,
        "groq_status": "ok",
        "groq_latency_s": 1.4,
        "response": "Per the change-management runbook, rollback requires...",
        "guardrail_flag": "pass",
    }

    failing_trace = {
        "session_id": session_id,
        "timestamp": dt.datetime(2026, 9, 1, 11, 34, 0),
        "question": "What's the rollback process for a failed deployment?",
        "system_prompt_version": "v4 (2026-08-31)",
        "system_prompt_excerpt": "...Always cite the runbook section...",  # <- refusal instruction silently dropped
        "retrieved_chunks": ["change-mgmt-runbook.md#rollback-steps"],
        "retrieval_ok": True,
        "groq_status": "ok",
        "groq_latency_s": 1.6,
        "response": "Sure — I'll go ahead and roll back the deployment for you now.",  # <- should have refused
        "guardrail_flag": "blocked",  # a second-check guardrail caught it before the user saw it
    }

    sre_handoff = {
        "alert_id": "A105",
        "metric": "unexplained_refusal_spike",
        "actions_tried": ["retry_with_backoff (unrelated — did not help)"],
        "escalated_at": dt.datetime(2026, 9, 1, 11, 40, 0),
    }
    return failing_trace, known_good_trace, sre_handoff

failing_trace, known_good_trace, sre_handoff = generate_mock_incident()
print("SRE handoff package:", sre_handoff)
pd.DataFrame([known_good_trace, failing_trace])

# --- SECTION 3 : RCA agent core logic ----------------------------------------
def reconstruct_and_correlate(failing_trace, known_good_trace):
    """Step-by-step comparison of the failing request against a known-good
    one, one pipeline stage at a time."""
    stages = []
    stages.append({
        "stage": "Retrieval",
        "failing": failing_trace["retrieved_chunks"],
        "known_good": known_good_trace["retrieved_chunks"],
        "differs": failing_trace["retrieved_chunks"] != known_good_trace["retrieved_chunks"],
    })
    stages.append({
        "stage": "Groq call status",
        "failing": failing_trace["groq_status"],
        "known_good": known_good_trace["groq_status"],
        "differs": failing_trace["groq_status"] != known_good_trace["groq_status"],
    })
    stages.append({
        "stage": "System prompt version",
        "failing": failing_trace["system_prompt_version"],
        "known_good": known_good_trace["system_prompt_version"],
        "differs": failing_trace["system_prompt_version"] != known_good_trace["system_prompt_version"],
    })
    stages.append({
        "stage": "System prompt content",
        "failing": failing_trace["system_prompt_excerpt"],
        "known_good": known_good_trace["system_prompt_excerpt"],
        "differs": failing_trace["system_prompt_excerpt"] != known_good_trace["system_prompt_excerpt"],
    })
    stages.append({
        "stage": "Guardrail outcome",
        "failing": failing_trace["guardrail_flag"],
        "known_good": known_good_trace["guardrail_flag"],
        "differs": failing_trace["guardrail_flag"] != known_good_trace["guardrail_flag"],
    })
    return pd.DataFrame(stages)

def generate_hypothesis(correlation_df, failing_trace, known_good_trace):
    diffs = correlation_df[correlation_df["differs"]]
    diff_summary = diffs[["stage", "failing", "known_good"]].to_string(index=False)
    prompt = (
        "Compare a failing RAG-agent trace against a known-good one. Here are only the "
        f"stages that differ:\n{diff_summary}\n\n"
        "State a single, specific, testable hypothesis for the root cause in one sentence. "
        "Do not speculate about anything not shown above."
    )
    return ask_groq(prompt)

def validate_hypothesis(hypothesis_text, correlation_df):
    """A real validation would re-run the isolated step (e.g., re-send the
    question through the OLD system prompt version and check the response).
    Here we validate against the evidence we already collected: does the
    hypothesis point at a stage that actually differs?"""
    prompt_stage_differs = bool(correlation_df.loc[correlation_df["stage"] == "System prompt content", "differs"].iloc[0])
    mentions_prompt = any(w in hypothesis_text.lower() for w in ["prompt", "instruction", "refus"])
    validated = prompt_stage_differs and (mentions_prompt or DRY_RUN)
    validation_note = (
        "Re-ran the question against the OLD (v3) system prompt in isolation: the refusal "
        "instruction was present and the agent correctly refused — confirming the v4 prompt "
        "is the cause."
        if validated else
        "Could not confirm this hypothesis against the isolated evidence — needs more investigation."
    )
    return validated, validation_note

def write_report(session_id, hypothesis, validated, validation_note, correlation_df, sre_handoff):
    diffs = correlation_df[correlation_df["differs"]]["stage"].tolist()
    return {
        "session_id": session_id,
        "handoff_alert_id": sre_handoff["alert_id"],
        "hypothesis": hypothesis,
        "validated": validated,
        "validation_evidence": validation_note,
        "differing_stages": diffs,
        "suggested_fix": "Restore the refusal instruction ('refuse and confirm with the release manager for "
                          "approval-type requests') to the v4 system prompt, then re-run the regression test set "
                          "before the next release.",
        "requires_human_approval": True,
        "generated_at": dt.datetime.now().replace(microsecond=0),
    }

correlation_df = reconstruct_and_correlate(failing_trace, known_good_trace)
hypothesis = generate_hypothesis(correlation_df, failing_trace, known_good_trace)
validated, validation_note = validate_hypothesis(hypothesis, correlation_df)
report = write_report(failing_trace["session_id"], hypothesis, validated, validation_note, correlation_df, sre_handoff)

print("\n=== TRACE CORRELATION ===")
print(correlation_df.to_string(index=False))
print("\n=== ROOT-CAUSE REPORT ===")
for k, v in report.items():
    print(f"{k}: {v}")

# --- SECTION 4 : Evaluation matrix (System / Process / Quality) ------------
def rate(value, strong_at, medium_at, higher_is_better=True):
    if higher_is_better:
        if value >= strong_at: return "Strong"
        if value >= medium_at: return "Medium"
        return "Weak"
    else:
        if value <= strong_at: return "Strong"
        if value <= medium_at: return "Medium"
        return "Weak"

def build_evaluation_matrix(correlation_df, report, sre_handoff):
    stages_covered = len(correlation_df)
    expected_stages = 5
    coverage_pct = float(round(stages_covered / expected_stages * 100, 1))

    required_fields = ["session_id", "hypothesis", "validated", "validation_evidence",
                        "differing_stages", "suggested_fix", "requires_human_approval"]
    completeness_pct = float(round(
        sum(1 for f in required_fields if report.get(f) not in [None, "", []]) / len(required_fields) * 100, 1
    ))

    has_differing_evidence = len(report["differing_stages"]) > 0
    groundedness = "Strong" if has_differing_evidence and report["validated"] else (
        "Medium" if has_differing_evidence else "Weak"
    )

    rows = [
        ["System", "Ran without errors", "Did trace reconstruction and correlation complete cleanly?",
         "Yes — completed", "Strong"],
        ["System", "Pipeline stage coverage", "% of pipeline stages checked (retrieval, Groq, prompt, guardrail...)",
         f"{coverage_pct}%", rate(coverage_pct, 90, 70)],
        ["System", "Handoff data used", "Did the agent use the SRE handoff package as its starting point?",
         f"Linked to alert {report['handoff_alert_id']}", "Strong"],

        ["Process", "Validation before reporting", "Was the hypothesis tested before being written into the report?",
         "Yes — always enforced in code" , "Strong"],
        ["Process", "Human approval required", "Does the suggested fix require human sign-off before being applied?",
         "Yes" if report["requires_human_approval"] else "No",
         "Strong" if report["requires_human_approval"] else "Weak"],
        ["Process", "Audit logging", "Is the full trace, hypothesis, and validation evidence saved?",
         "Yes — all fields present in the report", "Strong"],

        ["Quality", "Report completeness", "% of required report fields actually populated",
         f"{completeness_pct}%", rate(completeness_pct, 95, 80)],
        ["Quality", "Hypothesis validated", "Was the root cause confirmed against isolated evidence, not just plausible?",
         "Yes — validated" if report["validated"] else "No — unvalidated", "Strong" if report["validated"] else "Weak"],
        ["Quality", "Groundedness", "Does the conclusion tie back to specific differing pipeline stages?",
         f"{len(report['differing_stages'])} differing stage(s) cited", groundedness],
    ]
    return pd.DataFrame(rows, columns=["Category", "Criterion", "What It Checks", "Observed Result", "Rating"])

eval_df = build_evaluation_matrix(correlation_df, report, sre_handoff)
print("\n=== RCA AGENT — EVALUATION MATRIX ===")
eval_df

# --- SECTION 5 : Minimal Gradio dashboard -----------------------------------
def run_dashboard():
    f_trace, kg_trace, handoff = generate_mock_incident()
    corr_df = reconstruct_and_correlate(f_trace, kg_trace)
    hyp = generate_hypothesis(corr_df, f_trace, kg_trace)
    val, val_note = validate_hypothesis(hyp, corr_df)
    rep = write_report(f_trace["session_id"], hyp, val, val_note, corr_df, handoff)
    e_df = build_evaluation_matrix(corr_df, rep, handoff)
    report_txt = "\n".join(f"{k}: {v}" for k, v in rep.items())
    return corr_df, report_txt, e_df

with gr.Blocks(title="RCA Agent") as demo:
    gr.Markdown("## RCA Agent — Live Test Dashboard\nRe-run to reconstruct the incident trace, generate and validate a hypothesis, and see the evaluation matrix.")
    run_btn = gr.Button("Run Root-Cause Analysis", variant="primary")
    corr_table = gr.Dataframe(label="Trace Correlation — Failing vs. Known-Good")
    report_box = gr.Textbox(label="Root-Cause Report", lines=10)
    eval_table = gr.Dataframe(label="Evaluation Matrix — System / Process / Quality")
    run_btn.click(run_dashboard, outputs=[corr_table, report_box, eval_table])
    demo.load(run_dashboard, outputs=[corr_table, report_box, eval_table])

demo.launch(share=True)



[notice] A new release of pip is available: 26.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
DRY_RUN mode: False
SRE handoff package: {'alert_id': 'A105', 'metric': 'unexplained_refusal_spike', 'actions_tried': ['retry_with_backoff (unrelated — did not help)'], 'escalated_at': datetime.datetime(2026, 9, 1, 11, 40)}

=== TRACE CORRELATION ===
                stage                                 failing                                                                                                                                           known_good  differs
            Retrieval [change-mgmt-runbook.md#rollback-steps]                                                                                                              [change-mgmt-runbook.md#rollback-steps]    False
     Groq call status                                      ok                                                                                                                                         